In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [37]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [28]:
loader = PyPDFLoader("../data/internship_report.pdf")
docs = loader.load()
len(docs)

5

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
spllit_docs = splitter.split_documents(docs)
len(docs)
print(spllit_docs[:2])

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2026-08-14T01:47:33+00:00', 'author': 'Un-named', 'moddate': '2026-08-14T01:47:33+00:00', 'source': '../data/internship_report.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='D. Y. PATIL TECHNICAL CAMPUS, \nFACULTY OF ENGINEERING & FACULTY OF MANAGEMENT, TALSANDE \nDepartment of Computer Science & Engineering (Academic 2026-27) \nWeekly Internship Report \nStudent Name: Smartha Shesherao Radkar Roll No.:  \nCompany Name: Codexlabz Technologies, Atpadi, Sangli Place: Sangli \nWeek Duration: From: 8 June 2026   To: 13 June 2026 Reporting Week No.: 1 \nCompany Mentor: Ganesh Sutar \n \nWork Summary (Tasks/Activities Performed During the Week): \nSr. \nNo. Date Task / Activity Performed \n1 8 June Introduction to Data Analytics, data types, data analysis process and industry applications \n2 9 June Introduction to MS Excel and understanding worksheets, cells, rows, columns and data e

In [30]:
embedding = OllamaEmbeddings(model="all-minilm")


In [31]:
vector_store = Chroma.from_documents(
    documents=spllit_docs,
    embedding=embedding
)

In [32]:
query = "Student name"
data = vector_store.similarity_search(query=query)
len(data)


4

In [33]:
context = ""
for doc in data:
    context += data[0].page_content +"\n\n"

In [34]:
# print(context)
llm = ChatGroq(model="openai/gpt-oss-20b")

In [35]:
res = llm.invoke(f"Can you provide answer based on context for question , context : {context} and question :{query}")

In [36]:
print(res.content)

Smartha Shesherao Radkar


Chain - context_generate | prompt | llm | strparser

In [48]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content +  "\n\n"

    return {
        "context" : context,
        "question" : query
    }

In [49]:
prompt = PromptTemplate.from_template("""
    You are a assistant and provide answer based on context and 
    if you don't know the answer then say 'I dont know'
    Context : {context}
    Question : {question}
""")

In [50]:
rag_chain = get_context | prompt | llm

In [53]:
res = rag_chain.invoke("what is duration")

In [54]:
res.content

'The internship period covered in the reports runs from **8\u202fJune\u202f2026 to 5\u202fJuly\u202f2026**, which is a total of **28\u202fdays**.'